##### Copyright 2026 Google LLC.


In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Answer typed yes/no, choice, and score questions with the Gemini Decisions API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_decisions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The **Gemini Decision model** (`gemini-decision-1`) and **Decisions API** (`client.decisions` / `/v1beta/decisions`) answer one or more typed questions about your data in a single request. You send an `input` (text, JSON, images, audio, video, or PDFs) and a map of `questions`.

For every question, you get calibrated probabilities across outcomes alongside a top answer—a yes/no probability (`noul`), option probabilities and top selection (`choice`), or a probability-weighted rubric score and level probabilities (`score`)—plus a `confidence` score. Because nothing is generated, there is **no output text to parse and no output tokens to pay for** (`total_output_tokens = 0`).

Typical uses:
* Route a request to a model, queue, agent, or tool.
* Check inputs and outputs against many policy rules at once.
* Decide which tools an agent needs, or whether to compact, escalate, or stop.
* Filter, score, or rank large datasets against a rubric.
* React to UI state, audio, video frames, or documents in real time.


### Decisions API or structured output

| Use the **Decisions API** (`decisions.create`) when you need | Use the **Interactions API** (`interactions.create`) when you need |
| :--- | :--- |
| A yes/no probability (`noul`), one option from a fixed set (`choice`), or a rating on a fixed scale (`score`) | Generated text, summaries, or explanations |
| Calibrated probabilities and `confidence` scores you can threshold or sort | Extracted values or objects that follow your own JSON schema |
| Many independent checks over the same `input` in one call | Tool calls with arguments |
| Low latency and zero output token cost | Multi-step reasoning |


## Setup

### Install SDK

The Decisions API is supported in the official `google-genai` SDK.


In [ ]:
%pip install -U -q google-genai


### Setup your API key

To run the following cells, store your API key in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Authentication.ipynb) for a walkthrough.


In [ ]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)


### Choose a model

The Decisions API uses the `gemini-decision-1` model:


In [ ]:
MODEL_ID = "gemini-decision-1"  # @param ["gemini-decision-1"] {"allow-input": true, "isTemplate": true}


## 1. Quickstart: Ask a yes/no question (`noul`)

Every `client.decisions.create` request takes:
* `model`: The model ID (`gemini-decision-1`).
* `input`: Shared context (a `str`, `Content` dict, or list of `Content` blocks) evaluated once across all questions.
* `questions`: A map of caller-defined question keys (`[A-Za-z0-9_-]+`, up to 64 characters) to question definitions (`type`, `instructions`, and optional/required `criteria`).

Each question is evaluated independently against `input`—questions can't see each other, and adding or removing one doesn't change the other answers. The model never sees question keys, so write the full question in `instructions`.

In this example, you ask a single **`noul`** question—the Decisions API's yes/no probability type. It returns:
* `noul`: The calibrated probability `P(yes)` between `0.01` and `0.99` (so `P(no) = 1 - noul`).
* `confidence`: Certainty in the winning side (`max(noul, 1 - noul)`), between `0.50` and `0.99`.

> **Safety filtering (`prompt_feedback`):** When input safety filters block a request, the API returns HTTP 200 with `result.prompt_feedback` populated (`block_reason` and optional `block_reason_message`) and `result.answers` omitted. Check `result.answers` before indexing into it in production code.


In [ ]:
import json

input_text = "The blue mug I ordered on Monday arrived cracked on Wednesday."  # @param {type:"string"}
question_instructions = "Did the item arrive damaged?"  # @param {type:"string"}

result = client.decisions.create(
    model=MODEL_ID,
    input=input_text,
    questions={
        "is_damaged": {
            "type": "noul",
            "instructions": question_instructions,
        },
    },
)

answer = result.answers["is_damaged"]
print(
    f"P(yes) = {answer.noul:.2f} | P(no) = {1 - answer.noul:.2f} | "
    f"confidence = {answer.confidence:.2f}\n"
)
print(json.dumps(result.model_dump(), indent=2))


## 2. Question types (`noul`, `choice`, and `score`)

| Type | Use it for | `criteria` | Answer fields |
| :--- | :--- | :--- | :--- |
| **`noul`** | Yes/no checks and calibrated probability thresholds | Optional: `{"true_description": str, "false_description": str}` | `type`, `noul`, `confidence` |
| **`choice`** | Categorizing, routing, or ranking options in a fixed set | Required: map of option ID to description (`2` to `255` options) | `type`, `choice`, `probabilities`, `confidence` |
| **`score`** | Scoring, filtering, or ranking inputs against an ordered rubric | Required: list of non-empty level descriptions, lowest first (`2` to `255` levels) | `type`, `score`, `legend`, `probabilities`, `confidence` |


### 2.1 `noul`: yes or no

A **`noul`** (named for a calibrated boolean probability rather than a generated `"yes"`/`"no"` token) represents a binary yes/no decision.

**What a `noul` answer contains:**
* `type`: Always `"noul"`.
* `noul` (`float` in `[0.01, 0.99]`): The calibrated probability **`P(yes)`**. Because probabilities sum to `1.0`, **`P(no) = 1 - answer.noul`**.
* `confidence` (`float` in `[0.50, 0.99]`): The certainty of the leading outcome (`max(noul, 1 - noul)`).

**How to read and threshold `noul` values:**
* **Near `1.0` (for example, `noul >= 0.85`):** Strong **Yes** — both `noul` and `confidence` are high.
* **Near `0.0` (for example, `noul <= 0.15`):** Strong **No** — `noul` is low, `1 - noul` is high, and **`confidence` is still high** because the model is certain the answer is No.
* **Near `0.50` (`0.40` to `0.60`):** **Borderline / uncertain** — `confidence` is near `0.50`, so escalate to a human or run a deeper check.

> **Important:** `noul` is always clamped between `0.01` and `0.99`. Always keep thresholds strictly inside that range (checks like `noul > 0.99` or `noul < 0.01` never match).

Use optional `criteria` (`true_description` and `false_description`) to clarify edge cases when the question alone could be interpreted two ways:


In [ ]:
input_text = "Customer returned the sweater because they preferred green over blue."  # @param {type:"string"}
question_instructions = "Is the customer reporting a product defect?"  # @param {type:"string"}

result = client.decisions.create(
    model=MODEL_ID,
    input=input_text,
    questions={
        "is_defect": {
            "type": "noul",
            "instructions": question_instructions,
            "criteria": {
                "true_description": "The product itself is broken or faulty.",
                "false_description": "Preference, sizing, shipping, or pricing complaint.",
            },
        },
    },
)

answer = result.answers["is_defect"]
print(
    f"P(yes) = {answer.noul:.2f} | P(no) = {1 - answer.noul:.2f} | "
    f"confidence = {answer.confidence:.2f}\n"
)
print(json.dumps(answer.model_dump(), indent=2))


### 2.2 `choice`: pick one option

A `choice` question scores a set of options and returns both the winning option (`choice`) and the calibrated probability of every option (`probabilities`, summing to `1.0`), along with `confidence`. `criteria` maps each option ID to a description of when it applies (or `None` when the option ID is self-explanatory).

> **Alphabetical key sorting & out-of-scope inputs:**
> * Before evaluation, the API sorts `criteria` keys alphabetically (`UTF-8` byte order). Changing key order in your request doesn't change option positions, and exact probability ties break to the alphabetically first key.
> * `choice` always distributes probability across your options, even when none fits. Threshold `confidence` to catch low-certainty inputs, or pair the question with a separate `noul` check for whether the input is in scope (rather than relying on catch-all options like `other` or `none_of_the_above`).


In [ ]:
input_text = "I was charged twice for my order."  # @param {type:"string"}
question_instructions = "Which department should handle this request?"  # @param {type:"string"}

result = client.decisions.create(
    model=MODEL_ID,
    input=input_text,
    questions={
        "department": {
            "type": "choice",
            "instructions": question_instructions,
            "criteria": {
                "billing": "Charges, invoices, refunds, duplicate charges.",
                "shipping": "Tracking, delivery delays, missing packages.",
                "technical_support": "App crashes, device setup, login issues.",
                "sales": "Product recommendations, enterprise pricing, bulk orders.",
            },
        },
    },
)

answer = result.answers["department"]
print(f"Top choice : {answer.choice} (confidence = {answer.confidence:.2f})\n")
print("Probabilities:")
for dept, prob in answer.probabilities.items():
    print(f"  {dept:<20} {prob:.3f}")


### 2.3 `score`: rate on a numeric scale using text criteria

You might expect a `score` question to accept numbers like `[1, 2, 3, 4, 5]`. Instead, `criteria` takes an **ordered list of text descriptions** (`2` to `255` levels, lowest first; `2` to `10` works best for most rubrics):

* **Why text instead of numbers:** **Your list position (`0`, `1`, ..., `N - 1`) assigns the numeric scale automatically.** Passing qualitative rubric text lets the model anchor what `"0"` vs. `"3"` actually means, instead of guessing on an undefined number scale.
* **How the numeric `score` is computed:**
  1. **`probabilities`** (`dict[str, float]`): The model estimates a calibrated probability $p_i$ for every index `"0"`, `"1"`, ..., `str(N - 1)`, summing to `1.0`.
  2. **`score`** (`float` in `[0.00, N - 1.00]`): The **probability-weighted expected index** $\sum_i i \cdot p_i$. Because it averages across levels, `score` is continuous and can fall between integers (such as `2.08` when level `2` carries most probability with some mass on `3`).
  3. **`legend`** (`dict[str, str]`): Maps each string index (`"0"`, `"1"`, ...) back to your rubric description so you can look up what any level means.
  4. **`confidence`** (`float` in `[0, 1]`): How concentrated the distribution is on the leading level.

> **Tips:** Divide `answer.score / (len(criteria) - 1)` to normalize scores across rubrics of different lengths onto `[0.0, 1.0]`, or pick `max(answer.probabilities, key=answer.probabilities.get)` when you need a single discrete tier.


In [ ]:
input_text = "Export fails in Safari but works in Chrome."  # @param {type:"string"}

criteria = [
    "Trivial: typo or cosmetic defect with no functional impact.",
    "Low: workaround exists; non-blocking.",
    "Medium: core feature impaired for some users.",
    "High: core feature unusable for all users.",
    "Critical: data loss, security vulnerability, or system crash.",
]

result = client.decisions.create(
    model=MODEL_ID,
    input=input_text,
    questions={
        "bug_severity": {
            "type": "score",
            "instructions": "Rate the severity of this software bug.",
            "criteria": criteria,
        },
    },
)

answer = result.answers["bug_severity"]
top_level = max(answer.probabilities, key=answer.probabilities.get)
max_index = len(criteria) - 1
print(
    f"Expected score : {answer.score:.2f} / {max_index:.2f} "
    f"(normalized={answer.score / max_index:.2f})"
)
print(
    f"Top level [{top_level}]  : {answer.legend[top_level]} "
    f"(confidence={answer.confidence:.2f})\n"
)
print("Level probabilities:")
for level_idx in sorted(answer.probabilities, key=int):
    print(f"  [{level_idx}] {answer.probabilities[level_idx]:.3f} - {answer.legend[level_idx]}")
print("\nRaw answer object:")
print(json.dumps(answer.model_dump(), indent=2))


## 3. Ask multiple questions in one request

Evaluate multiple typed questions over the same `input` in a single request. Because the `input` is processed once across all questions, each additional question adds negligible latency and token overhead.

### 3.1 Composite customer support routing


In [ ]:
customer_message = "Audio cuts out on the left earcup. Fix it today or I cancel my subscription!"
order_context = {"tier": "platinum", "days_since_purchase": 12}
shared_input = json.dumps({"message": customer_message, "context": order_context}, indent=2)

noul_threshold = 0.50
confidence_threshold = 0.60

result = client.decisions.create(
    model=MODEL_ID,
    input=shared_input,
    questions={
        "is_defect": {
            "type": "noul",
            "instructions": "Is the user reporting physical hardware failure or defect?",
        },
        "next_action": {
            "type": "choice",
            "instructions": "What is the right resolution given `context`?",
            "criteria": {
                "instant_refund": "Refund without return. Low value or clear fault.",
                "replacement_shipment": "Ship a replacement. Customer returns the original.",
                "troubleshooting_flow": "The issue may be fixable by the customer.",
                "human_escalation": None,
            },
        },
        "churn_risk": {
            "type": "score",
            "instructions": "How likely is the customer to churn if the issue is not resolved now?",
            "criteria": [
                "No churn signal",
                "Mild frustration",
                "Mentions alternatives",
                "Threatens to cancel",
            ],
        },
    },
)

a = result.answers
if a["is_defect"].noul > noul_threshold and a["next_action"].confidence > confidence_threshold:
    action = a["next_action"].choice
else:
    action = "human_escalation"

next_act = a["next_action"]
print(f"is_defect P(yes) : {a['is_defect'].noul:.2f}")
print(f"next_action      : {next_act.choice} (confidence={next_act.confidence:.2f})")
print(f"churn_risk score : {a['churn_risk'].score:.2f} / 3.00")
print(f"--> Routed action: {action}")


### 3.2 Rank candidates and options in a single pass

Because `score` returns a continuous expectation and `choice` returns calibrated probabilities across options, you can rank retrieved documents or select top tools in one prefill pass:

1. **In-context candidate ranking (`score`):** Pass a query and candidate passages in `input`, ask one `score` question per candidate, and sort by `.score` descending.
2. **Top-K option ranking (`choice`):** Sort `.probabilities` descending to route to the top tools or categories.


In [ ]:
# 1. In-context candidate ranking: score 4 retrieved passages in a single pass
query = "How do I authenticate to the Gemini API using an environment variable?"
candidates = {
    "doc_1": (
        "Rate limits depend on your usage tier (Free, Tier 1, Tier 2) and are measured in RPM/TPM."
    ),
    "doc_2": (
        "Set the GEMINI_API_KEY environment variable in your shell, then initialize genai.Client() "
        "with no arguments; the SDK reads GEMINI_API_KEY automatically."
    ),
    "doc_3": (
        "You can pass an API key explicitly to genai.Client(api_key='YOUR_KEY') when "
        "initializing the client in Python."
    ),
    "doc_4": (
        "Use the Files API (client.files.upload) to upload large audio or video files before "
        "referencing them in requests."
    ),
}
relevance_rubric = [
    "Irrelevant: does not address the user's question.",
    "Tangential: mentions related concepts, but does not answer the question.",
    "Partial: answers a closely related question, but misses the specific requirement.",
    "Exact match: directly and completely answers the user's question.",
]

rank_result = client.decisions.create(
    model=MODEL_ID,
    input=json.dumps({"query": query, "candidates": candidates}, indent=2),
    questions={
        doc_id: {
            "type": "score",
            "instructions": f"How well does candidate `{doc_id}` answer `query`?",
            "criteria": relevance_rubric,
        }
        for doc_id in candidates
    },
)

print("Ranked candidates (by continuous score):")
for rank, (doc_id, ans) in enumerate(
    sorted(rank_result.answers.items(), key=lambda kv: kv[1].score, reverse=True), 1
):
    score_desc = f"score={ans.score:.2f}/3.00 (confidence={ans.confidence:.2f})"
    print(f"  #{rank} {doc_id}: {score_desc} -> {candidates[doc_id][:50]}...")

# 2. Top-K option ranking: sort `choice` probabilities to select top tools
user_task = (
    "Can you compare our Q3 cloud spend CSV against the Q2 invoice PDF and plot the variance?"
)
route_result = client.decisions.create(
    model=MODEL_ID,
    input=user_task,
    questions={
        "tool_route": {
            "type": "choice",
            "instructions": "Which specialist tool is most relevant to fulfill this user request?",
            "criteria": {
                "code_interpreter": (
                    "Run Python/pandas data analysis and generate charts from tabular data."
                ),
                "document_reader": "Extract tables and text from uploaded PDF documents.",
                "web_search": "Search the public web for live news or documentation.",
                "account_billing": (
                    "Modify subscription plans, payment methods, or billing addresses."
                ),
            },
        }
    },
)

print("\nTop-K tool routing (by calibrated probability):")
for rank, (tool_id, prob) in enumerate(
    sorted(
        route_result.answers["tool_route"].probabilities.items(),
        key=lambda kv: kv[1],
        reverse=True,
    ),
    1,
):
    print(f"  #{rank} {tool_id:<18} P={prob:.3f}")


## 4. Multimodal input and media resolution

`input` accepts `text`, `image`, `audio`, `video`, and `document` (`application/pdf`) content blocks using the same block schema as the Interactions API:
* Pass base64-encoded bytes in `data` (up to **100 MB per request**, or **50 MB for PDFs**) **or** a public HTTPS / signed URL in `uri` (up to **15 MB per file**, including PDFs) without uploading through the Files API.
* Files API URIs (`files/...` from `client.files.upload`) are not currently supported in `decisions.create`; use a public/signed HTTPS `uri` or base64 `data` instead.
* Always set `mime_type` to the real format (`application/octet-stream` is rejected).
* Set `resolution` (`"low"`, `"medium"`, `"high"`, or `"ultra_high"` on images only) on `image` and `video` blocks to control token usage per item:

| Media | Default | `resolution: "low"` |
| :--- | :--- | :--- |
| **Image** | ~1,090 tokens | ~260 tokens (`low`, `medium`, `high`, `ultra_high`) |
| **PDF page (`document`)** | ~520 tokens + extracted text | Not supported |
| **Video** | 1 fps (~66 image tokens + ~25 audio tokens/s) | ~66 image tokens/s (`low`, `medium`, `high`) |
| **Audio** | ~25 tokens/s | Not supported |


### 4.1 Images (base64 `data` and `resolution`)

Load a product return photo from the Cookbook media bucket, display it inline, and evaluate **three simultaneous questions** (`noul`, `choice`, and `score`) over the photo using base64 `data` and `resolution="low"`:


In [ ]:
import base64
from IPython.display import Image, display
import requests

image_url = "https://storage.googleapis.com/generativeai-downloads/images/damaged_blue_mug.jpg"
response = requests.get(image_url, timeout=30)
response.raise_for_status()
image_bytes = response.content
image_b64 = base64.b64encode(image_bytes).decode("utf-8")
image_resolution = "low"  # @param ["low", "medium", "high", "ultra_high"]

display(Image(data=image_bytes, width=480))

result = client.decisions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Inspect this customer return photo."},
        {
            "type": "image",
            "data": image_b64,
            "mime_type": "image/jpeg",
            "resolution": image_resolution,
        },
    ],
    questions={
        "is_damaged": {
            "type": "noul",
            "instructions": "Does the photo show physical damage to the product?",
        },
        "disposition": {
            "type": "choice",
            "instructions": "What warehouse disposition applies to this return?",
            "criteria": {
                "restock_new": "Item is unopened and in pristine condition.",
                "refurbish": "Minor cosmetic wear that can be cleaned or repackaged.",
                "scrap_and_refund": "Item is cracked, broken, or chipped beyond repair.",
            },
        },
        "damage_severity": {
            "type": "score",
            "instructions": "How severe is the damage visible on the item?",
            "criteria": [
                "None: intact with no visible damage.",
                "Minor: light surface scuff only, usable.",
                "Severe: structural crack, chip, or broken parts.",
            ],
        },
    },
)

disp = result.answers["disposition"]
print(f"is_damaged P(yes) : {result.answers['is_damaged'].noul:.2f}")
print(f"disposition       : {disp.choice} (confidence={disp.confidence:.2f})")
print(f"damage_severity   : {result.answers['damage_severity'].score:.2f} / 2.00")
print("\nToken usage by modality:")
print(json.dumps(result.usage.model_dump(), indent=2))


### 4.2 Audio, video, and PDFs (via public HTTPS `uri`)

You can pass any supported media block—including **PDF documents** (`type="document"` with `mime_type="application/pdf"`)—via a public HTTPS or signed `uri` (up to 15 MB per file) without uploading through the Files API. Evaluate an MP3 audio clip, an MP4 video clip at `resolution="low"`, and a PDF user guide:


In [ ]:
# 1. Audio evaluation via public HTTPS URI
audio_uri = "https://storage.googleapis.com/generativeai-downloads/data/Apollo-11_Day-01-Highlights-10s.mp3"
audio_question = "Is the speaker talking about spaceflight or a rocket launch?"

audio_result = client.decisions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "audio",
            "uri": audio_uri,
            "mime_type": "audio/mp3",
        },
    ],
    questions={
        "mentions_launch": {
            "type": "noul",
            "instructions": audio_question,
        },
    },
)
ans_audio = audio_result.answers["mentions_launch"]
print(
    f"Audio mentions_launch P(yes): {ans_audio.noul:.2f} "
    f"(confidence={ans_audio.confidence:.2f})"
)

# 2. Video evaluation via public HTTPS URI at low resolution
video_uri = "https://storage.googleapis.com/generativeai-downloads/videos/ForBiggerBlazes.mp4"
video_question = "Does the video show fire or flames?"
video_resolution = "low"  # @param ["low", "medium", "high"]

video_result = client.decisions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "video",
            "uri": video_uri,
            "mime_type": "video/mp4",
            "resolution": video_resolution,
        },
    ],
    questions={
        "shows_fire": {
            "type": "noul",
            "instructions": video_question,
        },
    },
)
ans_video = video_result.answers["shows_fire"]
print(
    f"Video shows_fire P(yes)     : {ans_video.noul:.2f} "
    f"(confidence={ans_video.confidence:.2f})"
)

# 3. PDF evaluation via public HTTPS URI (type="document", up to 15 MB)
pdf_uri = "https://storage.googleapis.com/generativeai-downloads/data/Nest-Cam-IQ-Outdoor_UserGuide-en-us.pdf"
pdf_question = "Does this document cover installation instructions for an outdoor camera?"

pdf_result = client.decisions.create(
    model=MODEL_ID,
    input=[
        {
            "type": "document",
            "uri": pdf_uri,
            "mime_type": "application/pdf",
        },
    ],
    questions={
        "is_camera_guide": {
            "type": "noul",
            "instructions": pdf_question,
        },
    },
)
ans_pdf = pdf_result.answers["is_camera_guide"]
print(
    f"PDF is_camera_guide P(yes)  : {ans_pdf.noul:.2f} "
    f"(confidence={ans_pdf.confidence:.2f})\n"
)
print("PDF token usage (pages are reported under 'image', extracted text under 'text'):")
print(json.dumps(pdf_result.usage.model_dump(), indent=2))


## 5. Best practices and request limits

### Write good questions

* **Keep each question focused on a single decision or rubric**: Complex rubrics work in one question when the criteria are explicit, and you can also split independent checks into separate questions in the same request and combine the answers in code.
* **Write the whole question in `instructions`**: The model never sees question keys (`"is_damaged"`, `"department"`) or your variable names.
* **Give each `choice` option and `score` level a short description**: Make adjacent `score` levels clearly different, and keep `score` scales linear (because `score` averages array indices `0, 1, 2, ...`, exponential scales like `1, 10, 100, 1000` skew the expected value).
* **Plan for inputs that fit no `choice` option**: Check `confidence` before acting on the top option, or pair the `choice` with a `noul` scope check (such as *"Is this message about an order?"*).
* **Prefer positive phrasing**: Ask *"Is the tier Gold?"* and compute `1 - noul` in code rather than asking negated questions such as *"Is the tier NOT Gold?"*.
* **Send only the fields the questions need, and compare numbers/dates in code**: Unrelated content lowers accuracy. Compare numbers, dates, and durations in code, then pass the result or a named bucket in `input`.
* **Treat `input` as untrusted**: Text in `input` can argue for its own classification. Write explicit criteria, test with adversarial examples, and keep a human or a second check on high-stakes actions such as payouts.


### Request limits

| Limit | Value |
| :--- | :--- |
| **Questions per request** | At least `1` (up to `1,000` recommended; bounded by the `262,144`-token context window) |
| **Options per `choice` question** | `2` to `255` |
| **Levels per `score` question** | `2` to `255` non-empty strings (`2` to `10` recommended for most rubrics) |
| **Question key** | `1` to `64` ASCII characters matching `[A-Za-z0-9_-]+` |
| **Option key** | `1` to `128` characters (up to `512` UTF-8 bytes; internal ASCII spaces allowed, no leading/trailing spaces or control chars) |
| **Context window** | `262,144` tokens across `input` and all questions |
| **Output token cost** | `0` (`total_output_tokens` is always `0`) |
| **Default rate limits** | Standard Gemini API Tier quotas apply (see [Rate limits](https://ai.google.dev/gemini-api/docs/rate-limits)) |


### Current limitations

* **Text-only question instructions**: Questions must be text; multimodal content belongs in `input`.
* **Media sources**: Pass images, audio, video, or PDFs (`type="document"`) as base64-encoded `data` (up to 50 MB for PDFs, 100 MB per request) or as a public/signed HTTPS `uri` (up to 15 MB per file). **Files API URIs (`files/...`)** and **YouTube links** are not currently supported in `decisions.create`.
* **No context caching, Batch API, or Flex tier**: Repeating the same `input` across requests counts it in full each time.


## Next steps

* Learn more about the Decisions API in the [Gemini API documentation](https://ai.google.dev/gemini-api/docs/decisions) and [Google AI Studio documentation](https://aistudio.google.com/docs/decisions-api).
* Explore related quickstarts:
  * [Interactions API](./Get_started_interactions_api.ipynb) to generate content, tool calls, and multi-turn conversations.
  * [JSON mode](./JSON_mode.ipynb) for extracting structured objects following custom Pydantic schemas.
  * [Prompting](./Prompting.ipynb) for advanced prompt engineering techniques.
